## PHASE 3.0 - DATA PREPARATION
### TASK 3.2: Cleaning Data

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

#Loading the original datasets
df = pd.read_csv(r"C:\Users\moyos\Downloads\Lendingclub.csv")
df_HC= pd.read_csv(r"C:\Users\moyos\Downloads\HomeCredit.csv")

C:\Users\moyos\AppData\Local\Temp\ipykernel_17644\1007284094.py:6: DtypeWarning: Columns (14) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(r"C:\Users\moyos\Downloads\Lendingclub.csv")


In [2]:
import json, numpy as np, pandas as pd

log = []
def record(dataset, step, n, note=''):
    log.append({'dataset': dataset, 'step': step, 'records_changed': int(n), 'note': note})

# Loading the presaved roles and raw data
lc_roles = json.load(open('lc_roles.json'))
hc_roles = json.load(open('hc_roles.json'))
lc_raw = pd.read_csv(r"C:\Users\moyos\Downloads\Lendingclub.csv")
hc_raw = pd.read_csv(r"C:\Users\moyos\Downloads\HomeCredit.csv")

C:\Users\moyos\AppData\Local\Temp\ipykernel_17644\4073599960.py:10: DtypeWarning: Columns (14) have mixed types. Specify dtype option on import or set low_memory=False.
  lc_raw = pd.read_csv(r"C:\Users\moyos\Downloads\Lendingclub.csv")


#### 3.1 selection

In [3]:

keep_lc = [c for k, v in lc_roles.items() if k != 'excluded' for c in v]
keep_hc = [c for k, v in hc_roles.items() if k != 'excluded' for c in v] + ['AMT_GOODS_PRICE']
lc = lc_raw[keep_lc].copy()
hc = hc_raw[keep_hc].copy()

#LendingClub Dataset
lc['issue_d'] = pd.to_datetime(lc['issue_d'], format='mixed')

bad = lc['dti_n'] > 100
assert bad.sum() == 533, bad.sum()
lc['dti_n_extreme'] = bad.astype(int)
lc.loc[bad, 'dti_n'] = np.nan
record('LC', 'dti_n > 100 to missing + flag', bad.sum(), '38 at 999, 495 other')

cap = lc['revenue'].quantile(0.999)
n_cap = (lc['revenue'] > cap).sum()
lc['revenue'] = lc['revenue'].clip(upper=cap)
record('LC', 'revenue winsorized at 99.9th pct', n_cap, f'cap = {cap:,.0f}')

miss = lc['zip_code'].isna()
lc.loc[miss, 'zip_code'] = 'unknown'
record('LC', 'zip_code missing to unknown', miss.sum())


In [5]:
#Home Credit Dataset
emp = hc['DAYS_EMPLOYED'] == 365243
hc['not_employed_flag'] = emp.astype(int)
hc.loc[emp, 'DAYS_EMPLOYED'] = np.nan
record('HC', 'DAYS_EMPLOYED placeholder to missing + flag', emp.sum())

xna = hc['ORGANIZATION_TYPE'] == 'XNA'
hc.loc[xna, 'ORGANIZATION_TYPE'] = 'Not applicable'
record('HC', 'ORGANIZATION_TYPE XNA to Not applicable', xna.sum())

occ_na = hc['OCCUPATION_TYPE'].isna()          # computed once, before any assignment
pens = hc['NAME_INCOME_TYPE'] == 'Pensioner'
hc.loc[occ_na & pens, 'OCCUPATION_TYPE'] = 'Not applicable (pensioner)'
hc.loc[occ_na & ~pens, 'OCCUPATION_TYPE'] = 'Unknown'
record('HC', 'OCCUPATION_TYPE missing split', occ_na.sum(),
       f'pensioner {(occ_na & pens).sum()}, unknown {(occ_na & ~pens).sum()}')

hc['ext3_missing'] = hc['EXT_SOURCE_3'].isna().astype(int)
record('HC', 'EXT_SOURCE_3 missing indicator added', hc['ext3_missing'].sum(), 'imputation deferred to pipeline')

print(hc['AMT_INCOME_TOTAL'].describe(percentiles=[.99, .999]).round(0))   # check the tail first
cap_i = hc['AMT_INCOME_TOTAL'].quantile(0.999)
n_i = (hc['AMT_INCOME_TOTAL'] > cap_i).sum()
hc['AMT_INCOME_TOTAL'] = hc['AMT_INCOME_TOTAL'].clip(upper=cap_i)
record('HC', 'AMT_INCOME_TOTAL winsorized at 99.9th pct', n_i, f'cap = {cap_i:,.0f}')



count       307511.0
mean        168798.0
std         237123.0
min          25650.0
50%         147150.0
99%         472500.0
99.9%       900000.0
max      117000000.0
Name: AMT_INCOME_TOTAL, dtype: float64


In [6]:
#Verification and log
assert len(lc) == 1347681 and len(hc) == 307511    
assert (hc['DAYS_EMPLOYED'] == 365243).sum() == 0
assert (lc['dti_n'] > 100).sum() == 0
assert not hc['OCCUPATION_TYPE'].isna().any() and not hc['ORGANIZATION_TYPE'].eq('XNA').any()

cleaning_log = pd.DataFrame(log)
print(cleaning_log.to_string(index=False))
cleaning_log.to_csv('cleaning_log.csv', index=False)

print(lc.isna().sum()[lambda s: s > 0])              
print(hc.isna().sum()[lambda s: s > 0])

lc.to_pickle('lc_clean.pkl'); hc.to_pickle('hc_clean.pkl')

dataset                                        step  records_changed                            note
     LC               dti_n > 100 to missing + flag              533            38 at 999, 495 other
     LC            revenue winsorized at 99.9th pct             1347                   cap = 585,000
     LC                 zip_code missing to unknown                1                                
     HC DAYS_EMPLOYED placeholder to missing + flag            55374                                
     HC     ORGANIZATION_TYPE XNA to Not applicable            55374                                
     HC               OCCUPATION_TYPE missing split            96391  pensioner 55357, unknown 41034
     HC        EXT_SOURCE_3 missing indicator added            60965 imputation deferred to pipeline
     HC   AMT_INCOME_TOTAL winsorized at 99.9th pct              278                   cap = 900,000
dti_n    533
dtype: int64
AMT_ANNUITY           12
DAYS_EMPLOYED      55374
EXT_SOURCE_2   

In [7]:
print("LENDINGCLUB")
print("Raw shape:", lc_raw.shape)
print("Cleaned shape:", lc.shape)
print("Duplicate rows:", lc.duplicated().sum())
print("Missing values:")
print(lc.isna().sum()[lambda s: s > 0])

print("\nHOME CREDIT")
print("Raw shape:", hc_raw.shape)
print("Cleaned shape:", hc.shape)
print("Duplicate rows:", hc.duplicated().sum())
print("Missing values:")
print(hc.isna().sum()[lambda s: s > 0])

LENDINGCLUB
Raw shape: (1347681, 15)
Cleaned shape: (1347681, 13)
Duplicate rows: 0
Missing values:
dti_n    533
dtype: int64

HOME CREDIT
Raw shape: (307511, 122)
Cleaned shape: (307511, 23)
Duplicate rows: 0
Missing values:
AMT_ANNUITY           12
DAYS_EMPLOYED      55374
EXT_SOURCE_2         660
EXT_SOURCE_3       60965
AMT_GOODS_PRICE      278
dtype: int64


In [8]:
# Inspect remaining missing values in LendingClub
print("LENDINGCLUB — dti_n missing records")
display(lc.loc[lc["dti_n"].isna()].head(10))

# Inspect remaining missing values in Home Credit
missing_hc = [
    "AMT_ANNUITY",
    "DAYS_EMPLOYED",
    "EXT_SOURCE_2",
    "EXT_SOURCE_3",
    "AMT_GOODS_PRICE"
]

for col in missing_hc:
    print(f"\n{col}: {hc[col].isna().sum()} missing")
    display(hc.loc[hc[col].isna()].head(5))


LENDINGCLUB — dti_n missing records


,revenue,dti_n,loan_amnt,fico_n,emp_length,purpose,home_ownership_n,Default,id,issue_d,addr_state,zip_code,dti_n_extreme
7767,62151.0,NaN,6550,672.0,NI,credit_card,MORTGAGE,0,67405134,2015-12-01,TX,773xx,1
9396,175000.0,NaN,15000,702.0,NI,debt_consolidation,MORTGAGE,0,67555887,2015-12-01,TX,774xx,1
20222,103000.0,NaN,20000,687.0,< 1 year,debt_consolidation,RENT,0,66495833,2015-12-01,MD,206xx,1
48971,116700.0,NaN,15000,752.0,NI,debt_consolidation,MORTGAGE,0,65571637,2015-12-01,CA,923xx,1
58351,270000.0,NaN,19000,677.0,1 year,other,MORTGAGE,0,64957302,2015-11-01,TX,774xx,1
64854,141200.0,NaN,12000,682.0,NI,debt_consolidation,RENT,0,64078746,2015-11-01,TX,773xx,1
68488,61465.0,NaN,12000,667.0,NI,debt_consolidation,MORTGAGE,0,63671798,2015-11-01,ME,049xx,1
85541,57000.0,NaN,5000,687.0,5 years,home_improvement,MORTGAGE,0,62206500,2015-10-01,KY,413xx,1
375641,136000.0,NaN,28000,702.0,< 1 year,debt_consolidation,RENT,1,130368940,2018-03-01,NV,895xx,1
375900,253600.0,NaN,37000,747.0,NI,debt_consolidation,MORTGAGE,0,130242528,2018-03-01,TX,774xx,1



AMT_ANNUITY: 12 missing


,NAME_CONTRACT_TYPE,AMT_CREDIT,AMT_ANNUITY,AMT_INCOME_TOTAL,NAME_INCOME_TYPE,OCCUPATION_TYPE,ORGANIZATION_TYPE,DAYS_EMPLOYED,NAME_EDUCATION_TYPE,NAME_HOUSING_TYPE,...,SK_ID_CURR,CODE_GENDER,DAYS_BIRTH,NAME_FAMILY_STATUS,REGION_POPULATION_RELATIVE,REGION_RATING_CLIENT,REGION_RATING_CLIENT_W_CITY,AMT_GOODS_PRICE,not_employed_flag,ext3_missing
47531,Cash loans,450000.0,NaN,180000.0,Commercial associate,High skill tech staff,Business Entity Type 3,-2523.0,Incomplete higher,House / apartment,...,155054,M,-10668,Single / not married,0.026392,2,2,450000.0,0,0
50035,Cash loans,450000.0,NaN,94500.0,Working,Laborers,Business Entity Type 1,-1270.0,Lower secondary,House / apartment,...,157917,F,-9027,Civil marriage,0.035792,2,2,450000.0,0,0
51594,Cash loans,539100.0,NaN,202500.0,Working,Cooking staff,Self-employed,-3950.0,Secondary / secondary special,House / apartment,...,159744,F,-17376,Married,0.046220,1,1,450000.0,0,0
55025,Cash loans,296280.0,NaN,162000.0,State servant,Core staff,Government,-2040.0,Higher education,House / apartment,...,163757,F,-11329,Married,0.035792,2,2,225000.0,0,0
59934,Cash loans,360000.0,NaN,202500.0,Working,Laborers,Other,-2498.0,Secondary / secondary special,House / apartment,...,169487,M,-19762,Single / not married,0.046220,1,1,360000.0,0,0



DAYS_EMPLOYED: 55374 missing


,NAME_CONTRACT_TYPE,AMT_CREDIT,AMT_ANNUITY,AMT_INCOME_TOTAL,NAME_INCOME_TYPE,OCCUPATION_TYPE,ORGANIZATION_TYPE,DAYS_EMPLOYED,NAME_EDUCATION_TYPE,NAME_HOUSING_TYPE,...,SK_ID_CURR,CODE_GENDER,DAYS_BIRTH,NAME_FAMILY_STATUS,REGION_POPULATION_RELATIVE,REGION_RATING_CLIENT,REGION_RATING_CLIENT_W_CITY,AMT_GOODS_PRICE,not_employed_flag,ext3_missing
8,Cash loans,1019610.0,33826.5,112500.000,Pensioner,Not applicable (pensioner),Not applicable,NaN,Secondary / secondary special,House / apartment,...,100011,F,-20099,Married,0.018634,2,2,913500.0,1,0
11,Cash loans,148365.0,10678.5,38419.155,Pensioner,Not applicable (pensioner),Not applicable,NaN,Secondary / secondary special,House / apartment,...,100015,F,-20417,Married,0.015221,2,2,135000.0,1,0
23,Cash loans,239850.0,23850.0,83250.000,Pensioner,Not applicable (pensioner),Not applicable,NaN,Secondary / secondary special,House / apartment,...,100027,F,-24827,Married,0.006296,3,3,225000.0,1,0
38,Cash loans,247275.0,17338.5,99000.000,Pensioner,Not applicable (pensioner),Not applicable,NaN,Secondary / secondary special,House / apartment,...,100045,F,-23920,Married,0.006207,2,2,225000.0,1,0
43,Cash loans,746280.0,42970.5,108000.000,Pensioner,Not applicable (pensioner),Not applicable,NaN,Higher education,House / apartment,...,100050,F,-23548,Single / not married,0.010966,2,2,675000.0,1,0


... (2 outputs left out)


EXT_SOURCE_3: 60965 missing


,NAME_CONTRACT_TYPE,AMT_CREDIT,AMT_ANNUITY,AMT_INCOME_TOTAL,NAME_INCOME_TYPE,OCCUPATION_TYPE,ORGANIZATION_TYPE,DAYS_EMPLOYED,NAME_EDUCATION_TYPE,NAME_HOUSING_TYPE,...,SK_ID_CURR,CODE_GENDER,DAYS_BIRTH,NAME_FAMILY_STATUS,REGION_POPULATION_RELATIVE,REGION_RATING_CLIENT,REGION_RATING_CLIENT_W_CITY,AMT_GOODS_PRICE,not_employed_flag,ext3_missing
1,Cash loans,1293502.5,35698.5,270000.0,State servant,Core staff,School,-1188.0,Higher education,House / apartment,...,100003,F,-16765,Married,0.003541,1,1,1129500.0,0,1
3,Cash loans,312682.5,29686.5,135000.0,Working,Laborers,Business Entity Type 3,-3039.0,Secondary / secondary special,House / apartment,...,100006,F,-19005,Civil marriage,0.008019,2,2,297000.0,0,1
4,Cash loans,513000.0,21865.5,121500.0,Working,Core staff,Religion,-3038.0,Secondary / secondary special,House / apartment,...,100007,M,-19932,Single / not married,0.028663,2,2,513000.0,0,1
9,Revolving loans,405000.0,20250.0,135000.0,Working,Laborers,Electricity,-2019.0,Secondary / secondary special,House / apartment,...,100012,M,-14469,Single / not married,0.019689,2,2,405000.0,0,1
14,Cash loans,773680.5,32778.0,189000.0,Working,Laborers,Transport: type 2,-203.0,Secondary / secondary special,House / apartment,...,100018,F,-14583,Married,0.010006,2,1,679500.0,0,1



AMT_GOODS_PRICE: 278 missing


,NAME_CONTRACT_TYPE,AMT_CREDIT,AMT_ANNUITY,AMT_INCOME_TOTAL,NAME_INCOME_TYPE,OCCUPATION_TYPE,ORGANIZATION_TYPE,DAYS_EMPLOYED,NAME_EDUCATION_TYPE,NAME_HOUSING_TYPE,...,SK_ID_CURR,CODE_GENDER,DAYS_BIRTH,NAME_FAMILY_STATUS,REGION_POPULATION_RELATIVE,REGION_RATING_CLIENT,REGION_RATING_CLIENT_W_CITY,AMT_GOODS_PRICE,not_employed_flag,ext3_missing
724,Revolving loans,135000.0,6750.0,45000.0,Commercial associate,Core staff,Kindergarten,-381.0,Secondary / secondary special,House / apartment,...,100837,F,-10072,Married,0.024610,2,2,NaN,0,1
5937,Revolving loans,450000.0,22500.0,157500.0,Working,Private service staff,Business Entity Type 3,-2921.0,Secondary / secondary special,House / apartment,...,106955,F,-11993,Married,0.010006,2,2,NaN,0,0
6425,Revolving loans,202500.0,10125.0,67500.0,Working,Accountants,Trade: type 7,-2712.0,Higher education,House / apartment,...,107494,F,-9727,Married,0.008474,2,2,NaN,0,1
6703,Revolving loans,180000.0,9000.0,121500.0,Working,Managers,Government,-899.0,Secondary / secondary special,House / apartment,...,107822,F,-11079,Civil marriage,0.011657,1,1,NaN,0,0
7647,Revolving loans,450000.0,22500.0,180000.0,Working,Security staff,Construction,-1847.0,Higher education,House / apartment,...,108913,M,-9986,Single / not married,0.032561,1,1,NaN,0,0


In [9]:

def compare_cap(raw_df, clean_df, column, target, label):
    original = raw_df[column]
    cap = original.quantile(0.999)
    affected = original > cap

    print(f"\n{label}: {column}")
    print(f"99.9th percentile cap: {cap:,.2f}")
    print(f"Records affected: {affected.sum():,}")

    print("\nDistribution before capping:")
    print(original.describe(
        percentiles=[0.50, 0.90, 0.99, 0.999]
    ).round(2))

    print("\nDistribution after capping:")
    print(clean_df[column].describe(
        percentiles=[0.50, 0.90, 0.99, 0.999]
    ).round(2))

    print("\nDefault rate among affected records:")
    print(raw_df.loc[affected, target].mean().round(4)
          if affected.any() else "No affected records")

    print("Default rate among unaffected records:")
    print(raw_df.loc[~affected, target].mean().round(4)
          if (~affected).any() else "No unaffected records")


compare_cap(
    lc_raw, lc, "revenue", "Default", "LendingClub"
)

compare_cap(
    hc_raw, hc, "AMT_INCOME_TOTAL", "TARGET", "Home Credit"
)



LendingClub: revenue
99.9th percentile cap: 585,000.00
Records affected: 1,347

Distribution before capping:
count     1347681.00
mean        77369.68
std         70362.99
min          1896.00
50%         65000.00
90%        129000.00
99%        260000.00
99.9%      585000.00
max      10999200.00
Name: revenue, dtype: float64

Distribution after capping:
count    1347681.00
mean       76824.81
std        49684.33
min         1896.00
50%        65000.00
90%       129000.00
99%       260000.00
99.9%     585000.00
max       585000.00
Name: revenue, dtype: float64

Default rate among affected records:
0.124
Default rate among unaffected records:
0.1999

Home Credit: AMT_INCOME_TOTAL
99.9th percentile cap: 900,000.00
Records affected: 278

Distribution before capping:
count    3.075110e+05
mean     1.687979e+05
std      2.371231e+05
min      2.565000e+04
50%      1.471500e+05
90%      2.700000e+05
99%      4.725000e+05
99.9%    9.000000e+05
max      1.170000e+08
Name: AMT_INCOME_TOTAL, dty

In [10]:

# LendingClub: highest reported incomes
print("LENDINGCLUB — highest revenue values")
display(
    lc_raw.nlargest(15, "revenue")[
        ["revenue", "Default", "loan_amnt", "fico_n", "dti_n"]
    ]
)

# Home Credit: highest reported incomes
print("HOME CREDIT — highest income values")
display(
    hc_raw.nlargest(15, "AMT_INCOME_TOTAL")[
        ["AMT_INCOME_TOTAL", "TARGET", "AMT_CREDIT",
         "AMT_ANNUITY", "NAME_INCOME_TYPE", "OCCUPATION_TYPE"]
    ]
)


LENDINGCLUB — highest revenue values


,revenue,Default,loan_amnt,fico_n,dti_n
401581,10999200.0,0,5000,687.0,0.07
574869,9550000.0,0,30000,807.0,0.07
974244,9522972.0,0,35000,712.0,0.32
203890,9500000.0,1,24000,727.0,0.12
902723,9300000.0,0,8400,672.0,0.02
497338,9225000.0,0,11600,667.0,0.11
35666,9000000.0,0,11000,707.0,0.08
191930,8900060.0,1,10550,692.0,0.09
356764,8706582.0,1,8000,687.0,0.11
21834,8700000.0,0,14000,687.0,0.18


HOME CREDIT — highest income values


,AMT_INCOME_TOTAL,TARGET,AMT_CREDIT,AMT_ANNUITY,NAME_INCOME_TYPE,OCCUPATION_TYPE
12840,117000000.0,1,562491.0,26194.5,Working,Laborers
203693,18000090.0,0,675000.0,69295.5,Commercial associate,NaN
246858,13500000.0,0,1400503.5,130945.5,Commercial associate,NaN
77768,9000000.0,0,1431531.0,132601.5,Working,Managers
131127,6750000.0,0,790830.0,52978.5,Working,Laborers
103006,4500000.0,0,2250000.0,225000.0,Working,Managers
187833,4500000.0,0,835380.0,42651.0,Working,Laborers
204564,4500000.0,0,450000.0,47749.5,Working,Managers
287463,4500000.0,0,1755000.0,61132.5,Working,Managers
181698,3950059.5,0,675000.0,66217.5,Commercial associate,Managers


In [11]:

thresholds = [900_000, 1_000_000, 2_000_000, 5_000_000]

for threshold in thresholds:
    group = hc_raw["AMT_INCOME_TOTAL"] > threshold

    print(f"\nIncome > {threshold:,.0f}")
    print("Records:", int(group.sum()))
    print("Default rate:",
          round(hc_raw.loc[group, "TARGET"].mean(), 4)
          if group.any() else "No records")
    print("Maximum income:",
          hc_raw.loc[group, "AMT_INCOME_TOTAL"].max()
          if group.any() else "N/A")



Income > 900,000
Records: 278
Default rate: 0.054
Maximum income: 117000000.0

Income > 1,000,000
Records: 250
Default rate: 0.052
Maximum income: 117000000.0

Income > 2,000,000
Records: 44
Default rate: 0.0455
Maximum income: 117000000.0

Income > 5,000,000
Records: 5
Default rate: 0.2
Maximum income: 117000000.0


## PHASE 3.0 - DATA PREPARATION
### TASK 3.3: Constructing Data

In [12]:
import numpy as np, pandas as pd

lc = pd.read_pickle('lc_clean.pkl'); hc = pd.read_pickle('hc_clean.pkl')
log33 = []
def made(ds, name, s):
    log33.append({'dataset': ds, 'attribute': name, 'non_missing': int(s.notna().sum()),
                  'missing': int(s.isna().sum()), 'min': round(float(s.min()), 3), 'max': round(float(s.max()), 3)})

def safe_ratio(a, b):
    return (a / b.where(b > 0)).replace([np.inf, -np.inf], np.nan)

In [13]:
#Home Credit 
hc['age_years'] = -hc['DAYS_BIRTH'] / 365.25
hc['age_band'] = pd.cut(hc['age_years'], [0, 30, 40, 50, 60, 200], right=False,
                        labels=['<30', '30-39', '40-49', '50-59', '60+'])
hc['credit_to_income']  = safe_ratio(hc['AMT_CREDIT'],  hc['AMT_INCOME_TOTAL'])
hc['annuity_to_income'] = safe_ratio(hc['AMT_ANNUITY'], hc['AMT_INCOME_TOTAL'])
hc['credit_minus_goods'] = hc['AMT_CREDIT'] - hc['AMT_GOODS_PRICE']
hc['goods_price_missing'] = hc['AMT_GOODS_PRICE'].isna().astype(int)
hc['employment_years'] = (-hc['DAYS_EMPLOYED'] / 365.25).clip(lower=0)

for c in ['age_years','credit_to_income','annuity_to_income','credit_minus_goods','employment_years']:
    made('HC', c, hc[c])

#Logging the other numeric attributes
made('HC', 'goods_price_missing', hc['goods_price_missing'])

# Log the categorical attribute separately
log33.append({
    'dataset': 'HC',
    'attribute': 'age_band',
    'non_missing': int(hc['age_band'].notna().sum()),
    'missing': int(hc['age_band'].isna().sum()),
    'min': None,
    'max': None
})

In [14]:
#LendingClub
lc['loan_to_income'] = safe_ratio(lc['loan_amnt'], lc['revenue'])
made('LC', 'loan_to_income', lc['loan_to_income'])

# Assertion Checks
assert len(lc) == 1347681 and len(hc) == 307511
assert hc['age_years'].between(18, 100).all()
assert hc['age_band'].notna().all()
assert not np.isinf(hc.select_dtypes('number')).any().any()
assert hc['credit_minus_goods'].isna().sum() == 278
assert hc['employment_years'].isna().sum() == 55374

In [15]:
print(pd.DataFrame(log33).to_string(index=False))
print(hc['age_band'].value_counts().sort_index())
print(hc.groupby('age_band', observed=True)['TARGET'].agg(['count','mean']).round(4))
print(hc[['credit_to_income','annuity_to_income','credit_minus_goods']].describe().round(3))
print(lc['loan_to_income'].describe().round(3))
print(lc.groupby(pd.qcut(lc['loan_to_income'], 10, duplicates='drop'), observed=True)['Default'].mean().round(4))
print(hc.groupby(pd.qcut(hc['credit_to_income'], 10, duplicates='drop'), observed=True)['TARGET'].mean().round(4))

pd.DataFrame(log33).to_csv('construction_log.csv', index=False)
lc.to_pickle('lc_built.pkl'); hc.to_pickle('hc_built.pkl')   # input for 3.5

dataset           attribute  non_missing  missing         min        max
     HC           age_years       307511        0      20.504     69.073
     HC    credit_to_income       307511        0       0.083     84.737
     HC   annuity_to_income       307499       12       0.007      1.876
     HC  credit_minus_goods       307233      278 -765000.000 540000.000
     HC    employment_years       252137    55374      -0.000     49.040
     HC goods_price_missing       307511        0       0.000      1.000
     HC            age_band       307511        0         NaN        NaN
     LC      loan_to_income      1347681        0       0.002      1.337
age_band
<30      45186
30-39    82331
40-49    76599
50-59    68094
60+      35301
Name: count, dtype: int64
          count    mean
age_band               
<30       45186  0.1144
30-39     82331  0.0959
40-49     76599  0.0764
50-59     68094  0.0612
60+       35301  0.0492
       credit_to_income  annuity_to_income  credit_minus_goods
co

In [16]:
print((hc['annuity_to_income'] > 1).sum(), (hc['credit_minus_goods'] < 0).sum(), (hc['credit_minus_goods'] == 0).sum())
print(hc.groupby(pd.qcut(hc['annuity_to_income'], 10, duplicates='drop'), observed=True)['TARGET'].mean().round(4))
print(hc.groupby(pd.qcut(hc['credit_minus_goods'].dropna(), 10, duplicates='drop'), observed=True)['TARGET'].mean().round(4))

33 260 108210
annuity_to_income
(0.0065, 0.08]    0.0714
(0.08, 0.104]     0.0726
(0.104, 0.125]    0.0782
(0.125, 0.144]    0.0785
(0.144, 0.163]    0.0788
(0.163, 0.186]    0.0838
(0.186, 0.212]    0.0853
(0.212, 0.247]    0.0885
(0.247, 0.302]    0.0884
(0.302, 1.876]    0.0819
Name: TARGET, dtype: float64
credit_minus_goods
(-765000.001, 0.0]      0.0675
(0.0, 17820.0]          0.0526
(17820.0, 39204.0]      0.0734
(39204.0, 60408.0]      0.0929
(60408.0, 88744.5]      0.0978
(88744.5, 117612.0]     0.0972
(117612.0, 161568.0]    0.0954
(161568.0, 540000.0]    0.0887
Name: TARGET, dtype: float64


In [17]:
# Count and summarize negative differences
negative = hc.loc[
    hc['credit_minus_goods'] < 0,
    ['AMT_CREDIT', 'AMT_GOODS_PRICE', 'credit_minus_goods', 'TARGET']
]

print("Negative differences:", len(negative))
print("\nSummary:")
print(negative['credit_minus_goods'].describe())

print("\nSample of 10 records:")
print(negative.head(10).to_string(index=False))

print("\nDefault rate: negative vs. non-negative difference")
print(
    hc.assign(
        negative_difference=hc['credit_minus_goods'] < 0
    ).groupby('negative_difference')['TARGET'].agg(
        records='count',
        default_rate='mean'
    ).round(4)
)

Negative differences: 260

Summary:
count       260.000000
mean    -179048.076923
std      157087.028593
min     -765000.000000
25%     -225000.000000
50%     -135000.000000
75%      -67500.000000
max      -22500.000000
Name: credit_minus_goods, dtype: float64

Sample of 10 records:
 AMT_CREDIT  AMT_GOODS_PRICE  credit_minus_goods  TARGET
   315000.0         450000.0           -135000.0       0
   135000.0         202500.0            -67500.0       0
   337500.0         540000.0           -202500.0       0
   135000.0         382500.0           -247500.0       0
   450000.0         900000.0           -450000.0       0
   157500.0         202500.0            -45000.0       0
   157500.0         180000.0            -22500.0       0
   225000.0         337500.0           -112500.0       0
   675000.0        1350000.0           -675000.0       0
   540000.0         900000.0           -360000.0       0

Default rate: negative vs. non-negative difference
                     records  default

## PHASE 3.0 - DATA PREPARATION
### TASK 3.4: Integrating Data

##### For the creditBridge project, both dataset are being treated independently

##### The two datasets are not merged because LendingClub and Home Credit measure different outcomes in different populations and share no identifier, so each is prepared and analyzed as its own dataset, and the framework is compared across them at the process level only. 

##### No internal joins are required: LendingClub is a single file, and Home Credit uses only application_train.csv, which holds one row per application.

In [18]:
import pandas as pd

lc = pd.read_pickle("lc_built.pkl")
hc = pd.read_pickle("hc_built.pkl")

print("LendingClub shape:", lc.shape)
print("Home Credit shape:", hc.shape)

print("LendingClub ID unique:", lc["id"].is_unique)
print("Home Credit ID unique:", hc["SK_ID_CURR"].is_unique)

print("LendingClub duplicate rows:", lc.duplicated().sum())
print("Home Credit duplicate rows:", hc.duplicated().sum())

print("Shared column names:", sorted(set(lc.columns) & set(hc.columns)))

LendingClub shape: (1347681, 14)
Home Credit shape: (307511, 30)
LendingClub ID unique: True
Home Credit ID unique: True
LendingClub duplicate rows: 0
Home Credit duplicate rows: 0
Shared column names: []


In [21]:
#Integrity checks
import pandas as pd
lc = pd.read_pickle('lc_built.pkl'); hc = pd.read_pickle('hc_built.pkl')
print(lc.shape, hc.shape)                                   # expect (1347681, 14) and (307511, 30)
print(lc['id'].is_unique, hc['SK_ID_CURR'].is_unique)       # True True
print(lc.duplicated().sum(), hc.duplicated().sum())         # 0 0
print(hc['employment_years'].min())                          # 0.0 after the clip fix
print(set(lc.columns) & set(hc.columns))                     # set()

(1347681, 14) (307511, 30)
True True
0 0
-0.0
set()


## PHASE 3.0 - DATA PREPARATION
### TASK 3.5: Formatting Data

In [30]:
# Reload the inputs so the cell can be rerun safely
lc = pd.read_pickle("lc_built.pkl").copy()
hc = pd.read_pickle("hc_built.pkl").copy()

log35 = []

def rec(ds, step, n, note=""):
    log35.append({"dataset": ds, "step": step, "records_changed": int(n), "note": note})

#LendingClub

emp_map = {
    "< 1 year": 0, "1 year": 1, "2 years": 2, "3 years": 3, "4 years": 4,
    "5 years": 5, "6 years": 6, "7 years": 7, "8 years": 8, "9 years": 9,
    "10+ years": 10
}

lc["emp_length"] = lc["emp_length"].astype("string").str.strip()
lc["emp_ni"] = (lc["emp_length"] == "NI").astype("int8")
lc["emp_length_years"] = lc["emp_length"].map(emp_map).astype("float64")

unmapped = lc["emp_length_years"].isna() & (lc["emp_length"] != "NI")
assert not unmapped.any(), (
    "Unmapped employment categories: "
    f"{lc.loc[unmapped, 'emp_length'].value_counts(dropna=False).to_dict()}"
)
assert lc["emp_length_years"].isna().sum() == lc["emp_ni"].sum() == 78188

rec("LC", "emp_length to ordered years; NI flagged", lc["emp_ni"].sum(),
    "NI stays missing in emp_length_years and is identified by emp_ni")
lc = lc.drop(columns="emp_length")

# Merge rare purpose categories
lc["purpose"] = lc["purpose"].astype("string")
counts = lc["purpose"].value_counts()
rare = counts[counts < 5000].index
assert set(rare) == {"educational", "renewable_energy", "wedding"}, set(rare)

rec("LC", "Rare purpose categories merged into rare_purpose", counts[rare].sum(),
    ", ".join(sorted(rare)))
lc["purpose"] = lc["purpose"].where(~lc["purpose"].isin(rare), "rare_purpose")

for c in ["purpose", "home_ownership_n", "addr_state"]:
    lc[c] = lc[c].astype("category")
lc["dti_n_extreme"] = lc["dti_n_extreme"].astype("int8")

#Home Credit

small = ["Maternity leave", "Businessman", "Student", "Unemployed"]
hc["NAME_INCOME_TYPE"] = hc["NAME_INCOME_TYPE"].astype("string")
n = hc["NAME_INCOME_TYPE"].isin(small).sum()
hc["NAME_INCOME_TYPE"] = hc["NAME_INCOME_TYPE"].where(
    ~hc["NAME_INCOME_TYPE"].isin(small), "Other")
rec("HC", "NAME_INCOME_TYPE tiny groups merged into Other", n, ", ".join(small))

#Academic degree is kept as its own category (164 records)

#Removing signed zero (-0.0) without touching other values
hc.loc[hc["employment_years"].eq(0), "employment_years"] = 0.0

# Drop raw columns replaced by constructed ones
hc = hc.drop(columns=["DAYS_EMPLOYED", "DAYS_BIRTH", "AMT_GOODS_PRICE"], errors="raise")

# Text columns to category (catches both object and string types)
for c in hc.select_dtypes(include=["object", "string"]).columns:
    if c != "SK_ID_CURR":
        hc[c] = hc[c].astype("category")

for c in ["not_employed_flag", "ext3_missing", "goods_price_missing"]:
    assert c in hc.columns, f"Expected flag is missing: {c}"
    hc[c] = hc[c].astype("int8")

#Final variable roles

final_roles = {
    "LC": {
        "predictors": ["revenue", "dti_n", "loan_amnt", "fico_n", "emp_length_years",
                       "emp_ni", "purpose", "home_ownership_n", "dti_n_extreme",
                       "loan_to_income"],
        "target": ["Default"],
        "identifier": ["id"],
        "time_validation": ["issue_d"],
        "audit": ["addr_state", "zip_code"]
    },
    "HC": {
        "predictors": ["NAME_CONTRACT_TYPE", "AMT_CREDIT", "AMT_ANNUITY",
                       "AMT_INCOME_TOTAL", "NAME_INCOME_TYPE", "OCCUPATION_TYPE",
                       "ORGANIZATION_TYPE", "NAME_EDUCATION_TYPE", "NAME_HOUSING_TYPE",
                       "EXT_SOURCE_2", "EXT_SOURCE_3", "employment_years",
                       "not_employed_flag", "ext3_missing", "credit_to_income",
                       "annuity_to_income", "credit_minus_goods", "goods_price_missing"],
        "target": ["TARGET"],
        "identifier": ["SK_ID_CURR"],
        "time_validation": [],
        "audit": ["CODE_GENDER", "NAME_FAMILY_STATUS", "REGION_POPULATION_RELATIVE",
                  "REGION_RATING_CLIENT", "REGION_RATING_CLIENT_W_CITY",
                  "age_years", "age_band"]
    }
}

for name, df_ in [("LC", lc), ("HC", hc)]:
    roles = final_roles[name]
    role_columns = [col for group in roles.values() for col in group]

    assert len(role_columns) == len(set(role_columns)), f"{name}: a column has multiple roles"
    assert set(role_columns) == set(df_.columns), (
        f"{name}: roles do not match columns. "
        f"No role: {set(df_.columns) - set(role_columns)}; "
        f"Not in data: {set(role_columns) - set(df_.columns)}")
    assert df_[roles["identifier"][0]].is_unique
    assert df_[roles["target"][0]].notna().all()
    assert df_[roles["target"][0]].isin([0, 1]).all()

assert lc.shape == (1347681, 15)
assert hc.shape == (307511, 27)
assert not np.isinf(lc.select_dtypes(include="number").to_numpy()).any()
assert not np.isinf(hc.select_dtypes(include="number").to_numpy()).any()

#Saving all the outputs

pd.DataFrame(log35).to_csv("formatting_log.csv", index=False)
with open("final_roles.json", "w") as f:
    json.dump(final_roles, f, indent=2)
lc.to_pickle("lc_final.pkl")
hc.to_pickle("hc_final.pkl")

print("LendingClub final shape:", lc.shape)
print("Home Credit final shape:", hc.shape)
print("\nFormatting log:")
print(pd.DataFrame(log35).to_string(index=False))
print("\nTask 3.5 structural checks passed.")

LendingClub final shape: (1347681, 15)
Home Credit final shape: (307511, 27)

Formatting log:
dataset                                             step  records_changed                                                             note
     LC          emp_length to ordered years; NI flagged            78188 NI stays missing in emp_length_years and is identified by emp_ni
     LC Rare purpose categories merged into rare_purpose             3709                           educational, renewable_energy, wedding
     HC   NAME_INCOME_TYPE tiny groups merged into Other               55                Maternity leave, Businessman, Student, Unemployed

Task 3.5 structural checks passed.


In [31]:
import pandas as pd, json
lc = pd.read_pickle('lc_final.pkl'); hc = pd.read_pickle('hc_final.pkl')
print(lc.shape, hc.shape)
print(pd.read_csv('formatting_log.csv').shape)
r = json.load(open('final_roles.json'))
print({k: {g: len(v) for g, v in d.items()} for k, d in r.items()})
print(hc['employment_years'].min(), lc['emp_length_years'].max())
print(hc['NAME_EDUCATION_TYPE'].value_counts().tail(2))

(1347681, 15) (307511, 27)
(3, 4)
{'LC': {'predictors': 10, 'target': 1, 'identifier': 1, 'time_validation': 1, 'audit': 2}, 'HC': {'predictors': 18, 'target': 1, 'identifier': 1, 'time_validation': 0, 'audit': 7}}
0.0 10.0
NAME_EDUCATION_TYPE
Lower secondary    3816
Academic degree     164
Name: count, dtype: int64
